In [ ]:
using CairoMakie; #Visualización en 3D
using LaTeXStrings; #Paquetería para utilizar texto en LaTeX en las gráficas
using DelimitedFiles; #Paquetería para leer y escribir archivos
using Printf;

AproxLambda(NSides) = (2π / (1 - cos(2π / NSides)));

DataPath = "Quasiperiodic-Tiles/Global Structural Studies/Data/SI_Fig9_SigmaSquare_gR_PerAreaGroup_1D/"; #Ruta para guardar los datos

function indice_Formula(x, NSides)
    if iseven(floor(NSides/2))
        if iseven(x)
            return Int((floor(NSides/2) + x) / 2)
        else
            return Int((ceil(NSides/2) - x) / 2)
        end
    else
        if iseven(x)
            return Int((ceil(NSides/2) - x) / 2)
        else
            return Int((floor(NSides/2) + x) / 2)
        end
    end
end

indice_Formula (generic function with 1 method)

### Visualización de la hiperuniformidad a partir de la N(R)

In [ ]:
###################################################################################################################
#                                   Datos del sistema cuasiperiódico 2D-1D
###################################################################################################################
NSides = 51;        #Simetría rotacional del sistema cuasiperiódico en 2D (Antes de la proyección)
Angulo = 0;         #Número del ángulo de la recta 1D a la cual se realiza la proyección de vectores 2D
Tamaño_Tesela = 26; #Ranking de la tesela (1 = Más pequeña)
Radio = 1800;       #Radio de la "vecindad circular" en 1D (Mitad del tamaño de la región cuadrada centrada en el origen)
###################################################################################################################
#                             Datos de los notebooks para la generación de datos
###################################################################################################################
Notebooks = 10;         #Número de Notebooks empleados
Vecindades = Int(1e4);  #Número de Vecindades por Notebook
###################################################################################################################
#                                   Datos para el cálculo de la N(R) y N^2(R)
###################################################################################################################
ΔR = 0.05;
Rango = 0.0:ΔR:Radio

for NSides in 27:2:27
    for Tamaño_Tesela in 1:Int(ceil(NSides/2))
        ###################################################################################################################
        #                                   Lectura de los datos de N(R) y N^2(R)
        ###################################################################################################################
        NR_Acumulado = vec(readdlm(DataPath * "NR_N$(NSides)_ThetaStarVectors$(Angulo)_Acumulados$(Vecindades)_DeltaStep0P05_Radius$(Radio)_Tesela$(Tamaño_Tesela)_Nb1.csv"));
        NR2_Acumulado = vec(readdlm(DataPath * "NR2_N$(NSides)_ThetaStarVectors$(Angulo)_Acumulados$(Vecindades)_DeltaStep0P05_Radius$(Radio)_Tesela$(Tamaño_Tesela)_Nb1.csv"));

        for Nb in 2:Notebooks
            NR_Acumulado .+= vec(readdlm(DataPath * "NR_N$(NSides)_ThetaStarVectors$(Angulo)_Acumulados$(Vecindades)_DeltaStep0P05_Radius$(Radio)_Tesela$(Tamaño_Tesela)_Nb$(Nb).csv"));
            NR2_Acumulado .+= vec(readdlm(DataPath * "NR2_N$(NSides)_ThetaStarVectors$(Angulo)_Acumulados$(Vecindades)_DeltaStep0P05_Radius$(Radio)_Tesela$(Tamaño_Tesela)_Nb$(Nb).csv"));
        end

        #Dividimos entre el número de vecindades acumuladas que se sumaron para obtener los datos
        NR_Acumulado = NR_Acumulado ./ (Vecindades * Notebooks);
        NR2_Acumulado = NR2_Acumulado ./ (Vecindades * Notebooks);
        σ2 = NR2_Acumulado .- (NR_Acumulado.^2);
        ###################################################################################################################
        #                                            Gráfica de la σ^2(R)    
        ###################################################################################################################
        Radio_Viz = 1050;
        if Tamaño_Tesela >= 3
            λ_i = indice_Formula(Tamaño_Tesela, NSides);                #Índice para el cálculo de la λ
            λ = abs(cos(2*λ_i*pi/NSides)*(NSides/2));                   #Longitud de escala del primer pico
            Radio_Viz = Int(ceil(14.1 * λ));
        end
        # --- Definición de las características del lienzo y las subgráficas en él ---
        Fig = Figure(size = (1800, 800)); #Lienzo en blanco donde se graficara
        Sigma2_Ax = Axis(
                        Fig[1, 1],                                                   #Posición en el lienzo donde se realizará la gráfica
                        title = L"Tile = %$(Tamaño_Tesela)",                         #Título de la gráfica
                        ylabel = L"\sigma^{2}(R)",                                   #Etiqueta que aparece en el eje vertical
                        titlesize = 55,                                              #Tamaño del título
                        xlabelsize = 55,                                             #Tamaño de la etiqueta al eje horizontal
                        ylabelsize = 55,                                             #Tamaño de la etiqueta al eje vertical
                        xticklabelsize = 40,                                         #Tamaño para el eje X
                        yticklabelsize = 40,                                         #Tamaño para el eje Y
                        xticksize = 25,                                              #Tamaño de los ticks horizontales
                        yticksize = 25,                                              #Tamaño de los ticks verticales
                        limits = ((ΔR, Radio_Viz), nothing),                         #Límites de la visualización para la gráfica
                        ytickformat = values -> [@sprintf("%.1f", v) for v in values]
                        )
        hidespines!(Sigma2_Ax, :t, :r); #Remueve las líneas de la caja que rodea a la gráfica ':t' = top, ':r' = right
        hidedecorations!(
                        Sigma2_Ax,
                        label = false,           #Se oculta o no las etiquetas a los ejes
                        ticklabels = false,      #Se oculta o no los valores de los ticks de los ejes
                        ticks = false            #Se oculta o no los ticks de los ejes
                        )
        # --- Gráfica de los datos de la σ^2(R) ---
        Final = 0; #Datos del final a eliminar (por errores en inconsistencias de tamaño en vecindades)
        if NSides == 25
            Final = 50;
        elseif NSides == 29 || NSides == 45
            Final = 170;
        end

        lines!(Sigma2_Ax, Rango[2:end - Final], σ2[1:end - Final])
        if Tamaño_Tesela >= 3
            # --- Líneas verticales con los múltiplos de la longitud de escala
            λ_i = indice_Formula(Tamaño_Tesela, NSides);                #Índice para el cálculo de la λ
            λ = abs(cos(2*λ_i*pi/NSides)*(NSides/2));                   #Longitud de escala del primer pico
            vlines!(Sigma2_Ax, [i*λ for i in 1:Int(floor(Radio/λ))],
                    linestyle = :dot,
                    alpha = 1,
                    linewidth = 5,
                    color = :red,
                    label = L"\lambda = \left|\cos \left( \frac{2*%$(λ_i)}{%$(NSides)} \right) * \frac{%$(NSides)}{2} \ \ \right| \approx %$(round(λ, digits = 2))",
                )
        end
        ###################################################################################################################
        #                                      Datos para el cálculo de la g(R)
        ###################################################################################################################
        ΔStep = 0.05;
        Rango = 0.0:ΔStep:Radio;
        ###################################################################################################################
        #                                        Lectura de los datos de g(R)
        ###################################################################################################################
        Pesos_Acumulado = vec(readdlm(DataPath * "gR_N$(NSides)_ThetaStarVectors$(Angulo)_Acumulados$(Vecindades)_DeltaStep0P05_Radius$(Radio)_Tesela$(Tamaño_Tesela)_Nb1.csv"));

        for Nb in 2:Notebooks
            Pesos_Acumulado .+= vec(readdlm(DataPath * "gR_N$(NSides)_ThetaStarVectors$(Angulo)_Acumulados$(Vecindades)_DeltaStep0P05_Radius$(Radio)_Tesela$(Tamaño_Tesela)_Nb$(Nb).csv"));
        end

        #Dividimos entre el número de vecindades acumuladas que se sumaron para obtener los datos
        Pesos_Acumulado = Pesos_Acumulado ./ (Vecindades * Notebooks);

        #Densidad de los ptos del decorado por vertices para factor de normalización
        Rho = (sum(Pesos_Acumulado[1:(end - Final)]) + 1)/(2 * Rango[end - Final]);
        FN = 1;

        ###ARREGLOS DE FRECUENCIAS DEL HISTOGRAMA Y MIDPOINTS DEL HISTOGRAMA
        #NOTA: El factor que multiplica a "Frecuencia0" consta de 2 términos. El primero (1/ΔR) es básicamente el inverso del 'Step' empleado en la variación del R para generar los datos originales de los que se obtuvo la altura
        #del histograma contenido en los archivos. El segundo término 1/(2 * Rho) es la normalización del histograma con respecto a la densidad de puntos que caen dentro de la caja de tamaño ΔR
        RangoMid0 = [(Rango[i] + Rango[i+1])/2 for i in 1:(length(Rango[1:(end - Final)])-1)]; #Pts medios de los bins en Histograma
        RangoNorm0 = FN*RangoMid0; #Aplicamos el factor de normalización a los valores del Radio
        Frecuencia0 = (Pesos_Acumulado[1:(end - Final)]) ./ (2 * Rho * ΔR); #Frecuencias de los histogramas normalizados

        ###VISUALIZACIÓN DE LOS DATOS
        Inicio0 = 1;    #Punto inicial a partir del cual se grafica. 
        Final0 = Final; #Punto final en el cual se deja de graficar

        #Informamos al usuario de los valores que toma la gráfica en los intervalos elegidos anteriormente
        RadioMin = RangoNorm0[Inicio0];                         #Mínimo valor de R que se graficará
        RadioMax = RangoNorm0[end-Final0];                      #Máximo valor de R que se graficará
        AlturaMin = minimum(Frecuencia0[Inicio0:(end-Final0)]); #Mínimo valor del Hist. asociado a la g(R)
        AlturaMax = maximum(Frecuencia0[Inicio0:(end-Final0)]); #Máximo valor del Hist. asociado a la g(R)
        ###################################################################################################################
        #                                            Gráfica de la g(R)    
        ###################################################################################################################
        # --- Definición de las características del lienzo y las subgráficas en él ---
        gR_Ax = Axis(
                    Fig[2, 1],
                    xlabel = L"R",                                               #Etiqueta que aparece en el eje horizontal
                    ylabel = L"g(R)",                                            #Etiqueta que aparece en el eje vertical
                    titlesize = 55,                                              #Tamaño del título
                    xlabelsize = 55,                                             #Tamaño de la etiqueta al eje horizontal
                    ylabelsize = 55,                                             #Tamaño de la etiqueta al eje vertical
                    xticklabelsize = 40,                                         #Tamaño para el eje X
                    yticklabelsize = 40,                                         #Tamaño para el eje Y
                    xticksize = 25,                                              #Tamaño de los ticks horizontales
                    yticksize = 25,                                              #Tamaño de los ticks verticales
                    limits = ((ΔR, Radio_Viz), nothing),                         #Límites de la visualización para la gráfica
                    ytickformat = values -> [@sprintf("%.1f", v) for v in values]
                    ) 
        hidespines!(gR_Ax, :t, :r); #Remueve las líneas de la caja que rodea a la gráfica ':t' = top, ':r' = right
        hidedecorations!(
                        gR_Ax,
                        label = false,           #Se oculta o no las etiquetas a los ejes
                        ticklabels = false,      #Se oculta o no los valores de los ticks de los ejes
                        ticks = false            #Se oculta o no los ticks de los ejes
                        )
        # --- Gráfica de los datos de la g(R) ---
        lines!(gR_Ax, RangoNorm0[Inicio0:(end - Final0)], Frecuencia0[Inicio0:(end - Final0)])
        if Tamaño_Tesela >= 3
            # --- Líneas verticales con los múltiplos de la longitud de escala
            λ_i = indice_Formula(Tamaño_Tesela, NSides);                #Índice para el cálculo de la λ
            λ = abs(cos(2*λ_i*pi/NSides)*(NSides/2));                   #Longitud de escala del primer pico
            vlines!(gR_Ax, [i*2*λ for i in 1:Int(floor(Radio/(2*λ)))],
                    linestyle = :dot,
                    alpha = 1,
                    linewidth = 5,
                    color = :red,
                    label = L"2 * \lambda = \left|\cos \left( \frac{2*%$(λ_i)}{%$(NSides)} \right) * %$(NSides) \ \ \right| \approx %$(round(2*λ, digits = 2))",
                )
        end

        yspace = maximum(tight_yticklabel_spacing!, [Sigma2_Ax, gR_Ax])

        Sigma2_Ax.yticklabelspace = yspace
        gR_Ax.yticklabelspace = yspace
        ###################################################################################################################
        #                                            Guardamos las gráficas    
        ###################################################################################################################
    end
end